# 🔗 Minería de Secuencias — Notebook Completo
> **Curso:** Machine Learning / Business Intelligence  
> **Objetivo:** Descubrir patrones ordenados temporalmente en sesiones web y secuencias de compras

---

## ¿Qué aprenderás?
1. La diferencia fundamental entre **Reglas de Asociación** y **Minería de Secuencias**
2. Preprocesar un **log de eventos crudo** hacia una base de datos de secuencias
3. Minar secuencias frecuentes con **PrefixSpan**
4. Calcular **Soporte Secuencial** y **Confianza Secuencial**
5. Construir e interpretar una **Matriz de Transición**
6. Aplicar los resultados a **análisis de embudo web** y **recomendaciones de cross-sell**

---
### ¿Por qué importa el orden?

| Técnica | Input | ¿Importa el orden? | Pregunta que responde |
|---|---|---|---|
| Reglas de Asociación | `{leche, pan, mantequilla}` | ❌ No | ¿Qué se compra **junto**? |
| Minería de Secuencias | `[home → products → cart → checkout]` | ✅ Sí | ¿Qué se hace **después** de qué? |

> **Ejemplo:** `[laptop → mouse]` y `[mouse → laptop]` son **dos patrones diferentes** en SM.  
> En AR, `{laptop, mouse}` es **el mismo itemset** sin importar el orden.

## 1. Instalación y Configuración

In [ ]:
!pip install prefixspan --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict
from prefixspan import PrefixSpan
import warnings
warnings.filterwarnings('ignore')

# ── Estilo de gráficos ────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#18181b',
    'axes.facecolor':   '#27272a',
    'axes.edgecolor':   '#3f3f46',
    'text.color':       '#e4e4e7',
    'axes.labelcolor':  '#a1a1aa',
    'xtick.color':      '#71717a',
    'ytick.color':      '#71717a',
    'grid.color':       '#3f3f46',
    'grid.alpha':        0.4,
    'axes.titlesize':    13,
    'axes.titlepad':     10,
})

BLUE    = '#3b82f6'
EMERALD = '#10b981'
AMBER   = '#f59e0b'
PURPLE  = '#8b5cf6'
PINK    = '#ec4899'
RED     = '#ef4444'
CYAN    = '#06b6d4'
ZINC    = '#71717a'

np.random.seed(42)
print('✅ Librerías cargadas correctamente')

---
## 2. Datasets

Trabajaremos con **dos contextos de negocio** distintos:

| # | Dataset | Unidad de secuencia | Evento | Pregunta de negocio |
|---|---|---|---|---|
| A | **Navegación Web** | Sesión de usuario | Página visitada | ¿Cuáles son los caminos al checkout? |
| B | **Historial de Compras** | Cliente | Categoría comprada | ¿Qué compra el cliente después de una laptop? |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DATASET A — Sesiones de Navegación Web (200 sesiones)
# ══════════════════════════════════════════════════════════════════

# Patrones de navegación con sus probabilidades
NAVIGATION_PATTERNS = [
    (['home', 'products', 'cart', 'checkout'],          0.18),
    (['home', 'products', 'checkout'],                  0.12),
    (['home', 'search', 'products', 'cart', 'checkout'],0.10),
    (['home', 'products', 'cart'],                      0.09),
    (['home', 'blog', 'products', 'cart', 'checkout'],  0.07),
    (['home', 'products', 'cart', 'payment', 'confirm'],0.07),
    (['home', 'about', 'contact'],                      0.05),
    (['home', 'search', 'products', 'checkout'],        0.06),
    (['home', 'login', 'products', 'cart', 'checkout'], 0.06),
    (['home', 'pricing', 'products', 'checkout'],       0.05),
    (['home', 'blog', 'about'],                         0.04),
    (['home', 'products', 'pricing', 'checkout'],       0.04),
    (['home', 'blog', 'products'],                      0.04),
    (['home', 'about', 'pricing', 'products'],          0.03),
]

def sample_sessions(patterns, n=200):
    probs = np.array([p for _, p in patterns])
    probs = probs / probs.sum()
    chosen = np.random.choice(len(patterns), size=n, p=probs)
    sessions = []
    for idx in chosen:
        base = patterns[idx][0].copy()
        # Añadir variación aleatoria ocasional
        if np.random.random() < 0.15:
            extra = np.random.choice(['search', 'blog', 'pricing', 'about'])
            insert_at = np.random.randint(1, max(1, len(base) - 1))
            base.insert(insert_at, extra)
        sessions.append(base)
    return sessions

web_sessions = sample_sessions(NAVIGATION_PATTERNS, 200)

# ══════════════════════════════════════════════════════════════════
# DATASET B — Historial de Compras de Clientes (150 clientes)
# ══════════════════════════════════════════════════════════════════

PURCHASE_PATTERNS = [
    (['laptop', 'mouse', 'keyboard', 'laptop_bag'],   0.14),
    (['phone', 'case', 'charger', 'earbuds'],          0.13),
    (['laptop', 'mouse', 'headphones'],                0.10),
    (['tablet', 'keyboard', 'stylus'],                 0.09),
    (['phone', 'charger', 'case', 'screen_protector'], 0.09),
    (['laptop', 'laptop_bag', 'mouse', 'keyboard'],    0.08),
    (['phone', 'earbuds', 'charger'],                  0.08),
    (['tablet', 'case', 'keyboard'],                   0.07),
    (['laptop', 'headphones', 'mouse'],                0.07),
    (['phone', 'case', 'charger', 'cable'],            0.06),
    (['laptop', 'mouse', 'keyboard', 'webcam'],        0.05),
    (['phone', 'charger', 'screen_protector'],         0.04),
]

purchase_sequences = sample_sessions(PURCHASE_PATTERNS, 150)

print(f'Dataset A — Sesiones web     : {len(web_sessions):4d} sesiones')
print(f'Dataset B — Compras clientes : {len(purchase_sequences):4d} secuencias')
print()
print('Ejemplos de sesiones web:')
for s in web_sessions[:3]:
    print(f'  {" → ".join(s)}')
print('Ejemplos de compras:')
for s in purchase_sequences[:3]:
    print(f'  {" → ".join(s)}')

---
## 3. Exploración de Datos (EDA)

In [ ]:
def eda_summary(sequences, name):
    lengths = [len(s) for s in sequences]
    all_events = [e for s in sequences for e in s]
    unique_events = sorted(set(all_events))
    event_counts = pd.Series(all_events).value_counts()
    event_support = event_counts / len(sequences)
    print(f'{'═'*55}')
    print(f'  {name}')
    print(f'{'═'*55}')
    print(f'  Secuencias totales   : {len(sequences)}')
    print(f'  Eventos únicos       : {len(unique_events)}')
    print(f'  Largo promedio       : {np.mean(lengths):.1f} eventos/secuencia')
    print(f'  Largo mín / máx      : {min(lengths)} / {max(lengths)}')
    print(f'  Densidad             : {sum(lengths)/(len(sequences)*len(unique_events)):.3f}')
    print(f'  Top 5 eventos        : {list(event_support.head(5).index)}')
    return event_support

web_support  = eda_summary(web_sessions,       'DATASET A — Navegación Web')
print()
purch_support = eda_summary(purchase_sequences, 'DATASET B — Compras de Clientes')

In [ ]:
# Colores por dominio
WEB_COLORS = {
    'home': EMERALD, 'products': BLUE, 'cart': AMBER, 'checkout': PURPLE,
    'search': CYAN, 'payment': '#a855f7', 'confirm': '#22c55e',
    'login': PINK, 'pricing': '#f97316', 'blog': ZINC, 'about': ZINC, 'contact': ZINC,
}
PURCH_COLORS = {
    'laptop': BLUE, 'mouse': EMERALD, 'keyboard': EMERALD, 'laptop_bag': AMBER,
    'headphones': PURPLE, 'webcam': CYAN, 'phone': '#f97316', 'case': PINK,
    'charger': '#a855f7', 'earbuds': '#22c55e', 'screen_protector': ZINC,
    'cable': ZINC, 'tablet': '#0ea5e9', 'stylus': AMBER,
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Soporte por Evento — Frecuencia Individual en Cada Dataset')

for ax, support, colors, title in [
    (axes[0], web_support,   WEB_COLORS,   'Páginas Web (Dataset A)'),
    (axes[1], purch_support, PURCH_COLORS, 'Categorías de Compra (Dataset B)'),
]:
    top = support.head(12)
    bar_colors = [colors.get(e, ZINC) for e in top.index]
    ax.barh(top.index[::-1], top.values[::-1], color=bar_colors[::-1], edgecolor='none', height=0.65)
    ax.set_xlabel('Soporte (fracción de secuencias)')
    ax.set_title(title)
    ax.grid(axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Distribución del largo de secuencias
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Distribución del Largo de Secuencias')

for ax, seqs, color, label in [
    (axes[0], web_sessions,       BLUE,    'Sesiones Web'),
    (axes[1], purchase_sequences, '#f97316','Secuencias de Compras'),
]:
    lengths = [len(s) for s in seqs]
    ax.hist(lengths, bins=range(1, max(lengths)+2), color=color, alpha=0.85,
            edgecolor='#18181b', rwidth=0.8)
    ax.axvline(np.mean(lengths), color=AMBER, linewidth=2, linestyle='--',
               label=f'media = {np.mean(lengths):.1f}')
    ax.set_xlabel('Largo de la secuencia (# eventos)')
    ax.set_ylabel('Frecuencia')
    ax.set_title(label)
    ax.legend()
    ax.grid(axis='y')

plt.tight_layout()
plt.show()

---
## 4. Preprocesamiento — Del Log Crudo a la Base de Datos de Secuencias

En la práctica, los datos **nunca llegan ya procesados**. Un sistema real te entrega un log como este:

```
session_id | timestamp           | page
S003       | 2024-03-01 14:02:01 | home
S001       | 2024-03-01 14:02:14 | home       ← entrelazado!
S003       | 2024-03-01 14:02:33 | products
S001       | 2024-03-01 14:02:45 | products
...
```

### Pipeline de preprocesamiento para SM
1. **Log crudo** → registros entrelazados con timestamps
2. **Agrupación por sesión** → ordenar eventos por timestamp por usuario
3. **Filtrado** → eliminar sesiones de largo 1
4. **Base de datos de secuencias** → lista de listas, lista para el algoritmo

> ⚠️ **Diferencia clave vs AR:** no se necesita codificación binaria. El orden se preserva tal cual.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 1 — Simular log de eventos crudo (entrelazado, con timestamps)
# ══════════════════════════════════════════════════════════════════
import datetime

def build_raw_log(sequences, session_prefix='S', start_time=None, interval_secs=(15, 180)):
    """Construye un log de eventos crudo a partir de secuencias ensambladas."""
    if start_time is None:
        start_time = datetime.datetime(2024, 3, 1, 9, 0, 0)
    records = []
    current_time = start_time
    for i, seq in enumerate(sequences[:10]):          # primeras 10 sesiones
        session_id = f'{session_prefix}{i+1:03d}'
        for event in seq:
            records.append({'session_id': session_id, 'timestamp': current_time, 'event': event})
            delta = np.random.randint(*interval_secs)
            current_time += datetime.timedelta(seconds=delta)
    # Mezclar registros (simular log real entrelazado)
    df = pd.DataFrame(records).sample(frac=1, random_state=7).reset_index(drop=True)
    return df

raw_log_web = build_raw_log(web_sessions, 'S')
print('PASO 1 — Log crudo (20 registros, entrelazados):')
print(raw_log_web.head(20).to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 2 — Agrupación por sesión y ordenamiento temporal
# ══════════════════════════════════════════════════════════════════
def assemble_sessions(raw_log):
    """Agrupa eventos por sesión y ordena por timestamp."""
    assembled = (
        raw_log
        .sort_values(['session_id', 'timestamp'])
        .groupby('session_id')['event']
        .apply(list)
        .reset_index()
    )
    assembled.columns = ['session_id', 'sequence']
    assembled['length'] = assembled['sequence'].apply(len)
    return assembled

assembled_web = assemble_sessions(raw_log_web)
print('PASO 2 — Sesiones ensambladas (agrupadas y ordenadas):')
for _, row in assembled_web.iterrows():
    arrow_seq = ' → '.join(row['sequence'])
    print(f"  {row['session_id']}  ({row['length']} eventos)  {arrow_seq}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 3 — Filtrado de secuencias demasiado cortas
# ══════════════════════════════════════════════════════════════════
def filter_sequences(assembled, min_length=2):
    before = len(assembled)
    filtered = assembled[assembled['length'] >= min_length].copy()
    removed  = before - len(filtered)
    print(f'Sesiones antes del filtrado : {before}')
    print(f'Eliminadas (largo < {min_length})      : {removed}')
    print(f'Sesiones conservadas        : {len(filtered)}')
    return filtered

# Añadir algunas sesiones de largo 1 artificialmente para demostrar el filtro
short_sessions = pd.DataFrame([
    {'session_id': 'S_X1', 'sequence': ['home'], 'length': 1},
    {'session_id': 'S_X2', 'sequence': ['products'], 'length': 1},
])
assembled_web_ext = pd.concat([assembled_web, short_sessions], ignore_index=True)

print('PASO 3 — Filtrado de secuencias:')
print()
filtered_web = filter_sequences(assembled_web_ext, min_length=2)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 4 — Base de datos de secuencias (formato final)
# ══════════════════════════════════════════════════════════════════
# Para los algoritmos de SM, el formato es simplemente una lista de listas
# NO se necesita matriz binaria ni codificación numérica

web_db     = web_sessions       # Dataset A completo (ya limpio)
purchase_db = purchase_sequences # Dataset B completo

print('PASO 4 — Formato final de la base de datos de secuencias:')
print('(lista de listas — cada sublista es una sesión ordenada)')
print()
print('web_db[:5] =')
for seq in web_db[:5]:
    print(f'  {seq}')
print()
print('purchase_db[:5] =')
for seq in purchase_db[:5]:
    print(f'  {seq}')
print()
print('✅ Listo para PrefixSpan — sin binarización, sin encoding numérico')

---
## 5. Minería con PrefixSpan

### ¿Cómo funciona PrefixSpan?
1. Encuentra todos los **prefijos frecuentes** de largo 1
2. Para cada prefijo `α`, construye la **base de datos proyectada**: solo los **sufijos** de secuencias que contienen `α`
3. Mina recursivamente la BD proyectada para extender el prefijo
4. Garantía: **cada patrón descubierto es frecuente por construcción** (no se generan candidatos falsos)

```
BD proyectada para [home]:
  S001: [products, cart, checkout]  ← sufijo después de 'home'
  S002: [products, checkout]
  S003: [search, products, cart, checkout]
  ...
```

In [ ]:
# ── Función auxiliar para proyección ─────────────────────────────
def get_projected_db(sequences, prefix_event, show=6):
    """Muestra la BD proyectada para un prefijo dado."""
    print(f'Base de datos proyectada para prefijo [{prefix_event}]:')
    print(f'{"─"*55}')
    count = 0
    for i, seq in enumerate(sequences):
        if prefix_event in seq:
            idx = seq.index(prefix_event)
            suffix = seq[idx+1:]
            if count < show:
                suffix_str = ' → '.join(suffix) if suffix else '∅ (vacío)'
                print(f'  S{i+1:03d}: [{prefix_event}] | {suffix_str}')
            count += 1
    if count > show:
        print(f'  ... ({count - show} sesiones más)')
    print(f'Total secuencias con [{prefix_event}]: {count} → soporte = {count/len(sequences):.2f}')

get_projected_db(web_db, 'home')
print()
get_projected_db(web_db, 'products')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Minar Dataset A — Navegación Web
# ══════════════════════════════════════════════════════════════════
MIN_SUPPORT_WEB = 0.30   # ← prueba con 0.20, 0.40, 0.50
MAX_LEN_WEB     = 4

ps_web = PrefixSpan(web_db)
min_count_web = int(MIN_SUPPORT_WEB * len(web_db))
results_web_raw = ps_web.frequent(min_count_web, closed=False)

# Convertir a DataFrame
df_web = pd.DataFrame(
    [(count, pattern) for count, pattern in results_web_raw if len(pattern) <= MAX_LEN_WEB],
    columns=['count', 'pattern']
)
df_web['support']  = df_web['count'] / len(web_db)
df_web['length']   = df_web['pattern'].apply(len)
df_web['pattern_str'] = df_web['pattern'].apply(lambda p: ' → '.join(p))
df_web = df_web.sort_values('support', ascending=False)

print(f'Dataset A — Web Navigation')
print(f'Patrones frecuentes encontrados (soporte ≥ {MIN_SUPPORT_WEB}): {len(df_web)}')
print()
print(df_web[['pattern_str', 'support', 'length']].head(15).to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Minar Dataset B — Historial de Compras
# ══════════════════════════════════════════════════════════════════
MIN_SUPPORT_PURCH = 0.25
MAX_LEN_PURCH     = 4

ps_purch = PrefixSpan(purchase_db)
min_count_purch = int(MIN_SUPPORT_PURCH * len(purchase_db))
results_purch_raw = ps_purch.frequent(min_count_purch, closed=False)

df_purch = pd.DataFrame(
    [(count, pattern) for count, pattern in results_purch_raw if len(pattern) <= MAX_LEN_PURCH],
    columns=['count', 'pattern']
)
df_purch['support']     = df_purch['count'] / len(purchase_db)
df_purch['length']      = df_purch['pattern'].apply(len)
df_purch['pattern_str'] = df_purch['pattern'].apply(lambda p: ' → '.join(p))
df_purch = df_purch.sort_values('support', ascending=False)

print(f'Dataset B — Historial de Compras')
print(f'Patrones frecuentes encontrados (soporte ≥ {MIN_SUPPORT_PURCH}): {len(df_purch)}')
print()
print(df_purch[['pattern_str', 'support', 'length']].head(15).to_string(index=False))

---
## 6. Métricas Secuenciales

### Soporte Secuencial
$$\text{support}([A \to B]) = \frac{|\{\text{secuencias que contienen A antes de B}\}|}{|\text{total de secuencias}|}$$

**Independiente del gap**: `[A...B]` cuenta aunque haya otros eventos entre A y B.

### Confianza Secuencial
$$\text{conf}([A] \Rightarrow [B]) = \frac{\text{support}([A \to B])}{\text{support}([A])}$$

Dado que ocurrió A, ¿qué tan probable es que ocurra B después?

In [ ]:
def is_subsequence(pattern, sequence):
    """Verifica si 'pattern' es una subsecuencia de 'sequence' (gap-free no requerido)."""
    pi = 0
    for event in sequence:
        if pi < len(pattern) and event == pattern[pi]:
            pi += 1
    return pi == len(pattern)

def seq_support(pattern, sequences):
    count = sum(1 for s in sequences if is_subsequence(pattern, s))
    return count / len(sequences)

def seq_confidence(antecedent, consequent_event, sequences):
    """conf([antecedent] => [consequent_event]) — busca el evento después del antecedente."""
    pattern_together = antecedent + [consequent_event]
    sup_together = seq_support(pattern_together, sequences)
    sup_ant      = seq_support(antecedent, sequences)
    return sup_together / sup_ant if sup_ant > 0 else 0

# ── Verificación manual — Dataset A ──────────────────────────────
print('=== MÉTRICAS DATASET A — Navegación Web ===')
test_cases_web = [
    (['home'], 'products'),
    (['home'], 'checkout'),
    (['home', 'products'], 'cart'),
    (['home', 'products', 'cart'], 'checkout'),
    (['products', 'cart'], 'checkout'),
]
for ant, con in test_cases_web:
    sup  = seq_support(ant + [con], web_db)
    conf = seq_confidence(ant, con, web_db)
    print(f'  [{" → ".join(ant)}] ⟹ [{con}]')
    print(f'    soporte = {sup:.3f}  |  confianza = {conf:.3f}')
    print()

print()
print('=== MÉTRICAS DATASET B — Historial de Compras ===')
test_cases_purch = [
    (['laptop'], 'mouse'),
    (['phone'],  'charger'),
    (['laptop', 'mouse'], 'keyboard'),
    (['phone', 'charger'], 'earbuds'),
]
for ant, con in test_cases_purch:
    sup  = seq_support(ant + [con], purchase_db)
    conf = seq_confidence(ant, con, purchase_db)
    print(f'  [{" → ".join(ant)}] ⟹ [{con}]')
    print(f'    soporte = {sup:.3f}  |  confianza = {conf:.3f}')
    print()

---
## 7. Matriz de Transición

La **confianza secuencial entre pares de eventos** puede visualizarse como una matriz de transición:

$$M[A][B] = P(B \text{ ocurre después de } A \text{ en la misma secuencia})$$

Esta matriz es la base de los **modelos de Markov de primer orden** para navegación web.

In [ ]:
def build_transition_matrix(sequences, top_n=10):
    """Construye la matriz de probabilidades de transición entre los top_n eventos."""
    all_events = [e for s in sequences for e in s]
    top_events = pd.Series(all_events).value_counts().head(top_n).index.tolist()
    n = len(sequences)
    matrix = pd.DataFrame(0.0, index=top_events, columns=top_events)
    for from_ev in top_events:
        for to_ev in top_events:
            count = sum(1 for seq in sequences
                        if from_ev in seq and to_ev in seq[seq.index(from_ev)+1:])
            matrix.loc[from_ev, to_ev] = count / n
    return matrix

# ── Dataset A: Navegación Web ─────────────────────────────────────
print('Calculando matriz de transición para navegación web...')
transition_web   = build_transition_matrix(web_db, top_n=9)
print('Calculando matriz de transición para compras...')
transition_purch = build_transition_matrix(purchase_db, top_n=10)
print('✅ Listo')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Matrices de Confianza Secuencial\n(celda [i,j] = P(columna j ocurre después de fila i))')

for ax, matrix, title in [
    (axes[0], transition_web,   'Dataset A — Navegación Web'),
    (axes[1], transition_purch, 'Dataset B — Compras de Clientes'),
]:
    # Enmascarar diagonal
    mask = np.eye(len(matrix), dtype=bool)
    sns.heatmap(
        matrix, ax=ax, mask=mask,
        cmap='Blues', vmin=0, vmax=1,
        linewidths=0.4, linecolor='#18181b',
        annot=True, fmt='.2f', annot_kws={'size': 9}
    )
    ax.set_title(title)
    ax.set_xlabel('Evento siguiente (columna)')
    ax.set_ylabel('Evento desde (fila)')
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right')

plt.tight_layout()
plt.show()

print('Interpretación: valores altos (azul oscuro) = alta probabilidad de que la columna')
print('ocurra DESPUÉS de la fila. Ej: P(checkout | home → products → cart) es la confianza secuencial.')

---
## 8. Visualizaciones

In [ ]:
# ── Top patrones por soporte — Dataset A ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Top Patrones Frecuentes por Soporte')

for ax, df, color, n_top, title in [
    (axes[0], df_web.query('length >= 2'),   BLUE,     12, 'Dataset A — Navegación Web'),
    (axes[1], df_purch.query('length >= 2'), '#f97316', 12, 'Dataset B — Compras de Clientes'),
]:
    top = df.nlargest(n_top, 'support')
    ax.barh(top['pattern_str'][::-1], top['support'][::-1],
            color=color, alpha=0.85, edgecolor='none', height=0.65)
    ax.set_xlabel('Soporte')
    ax.set_title(title)
    ax.grid(axis='x')
    # Anotar largo del patrón
    for _, row in top.iterrows():
        ax.text(row['support'] + 0.005, list(top['pattern_str'])[::-1].index(row['pattern_str']),
                f"L={row['length']}", va='center', fontsize=7, color=AMBER)

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribución de patrones por largo ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Número de Patrones Encontrados por Largo de Secuencia')

for ax, df, color, title in [
    (axes[0], df_web,   BLUE,     'Dataset A — Web'),
    (axes[1], df_purch, '#f97316', 'Dataset B — Compras'),
]:
    len_counts = df.groupby('length').size()
    ax.bar(len_counts.index, len_counts.values, color=color, alpha=0.85, edgecolor='#18181b')
    ax.set_xlabel('Largo del patrón')
    ax.set_ylabel('Número de patrones')
    ax.set_title(title)
    ax.set_xticks(len_counts.index)
    ax.grid(axis='y')
    for x, y in zip(len_counts.index, len_counts.values):
        ax.text(x, y + 0.3, str(y), ha='center', fontsize=9, color=AMBER)

plt.tight_layout()
plt.show()

In [ ]:
# ── Efecto del umbral min_support en el número de patrones ───────
thresholds_web   = np.arange(0.10, 0.70, 0.05)
thresholds_purch = np.arange(0.10, 0.60, 0.05)

def count_patterns(db, thresholds, max_len=4):
    ps = PrefixSpan(db)
    counts = []
    for t in thresholds:
        mc = int(t * len(db))
        res = ps.frequent(mc, closed=False)
        counts.append(sum(1 for c, p in res if len(p) <= max_len))
    return counts

print('Calculando curvas de sensibilidad...')
counts_web   = count_patterns(web_db, thresholds_web)
counts_purch = count_patterns(purchase_db, thresholds_purch)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Sensibilidad al Umbral de Soporte')

for ax, thrs, cnts, color, cur_t, title in [
    (axes[0], thresholds_web,   counts_web,   BLUE,     MIN_SUPPORT_WEB,   'Dataset A — Web'),
    (axes[1], thresholds_purch, counts_purch, '#f97316', MIN_SUPPORT_PURCH, 'Dataset B — Compras'),
]:
    ax.plot(thrs, cnts, color=color, linewidth=2.5, marker='o', markersize=5)
    ax.fill_between(thrs, cnts, alpha=0.15, color=color)
    ax.axvline(cur_t, color=AMBER, linewidth=1.5, linestyle='--', label=f'umbral actual = {cur_t}')
    ax.set_xlabel('min_support')
    ax.set_ylabel('Número de patrones (largo ≤ 4)')
    ax.set_title(title)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

---
## 9. Análisis de Negocios — Dataset A: Embudo Web

En navegación web, los patrones secuenciales permiten responder:
- ¿Cuáles son los caminos más frecuentes hacia el `checkout`?
- ¿Dónde se pierden los usuarios (drop-off)?
- ¿Qué páginas intermedias aumentan la probabilidad de conversión?

In [ ]:
# ── Análisis de embudo de conversión ─────────────────────────────
FUNNEL = ['home', 'products', 'cart', 'checkout', 'payment', 'confirm']

print('🔽 EMBUDO DE CONVERSIÓN — Tasas de retención por etapa')
print('═' * 60)

funnel_data = []
for step in FUNNEL:
    count = sum(1 for s in web_db if step in s)
    sup   = count / len(web_db)
    funnel_data.append({'step': step, 'count': count, 'support': sup})
    print(f'  {step:12s}  {count:4d} sesiones  ({sup:.0%})')

print()
print('Tasas de conversión entre etapas:')
for i in range(1, len(funnel_data)):
    prev  = funnel_data[i-1]
    curr  = funnel_data[i]
    rate  = curr['count'] / prev['count'] if prev['count'] > 0 else 0
    drop  = 1 - rate
    print(f'  {prev["step"]:12s} → {curr["step"]:12s}: '
          f'{rate:.0%} conversión  |  {drop:.0%} drop-off')

In [ ]:
# Visualizar embudo
df_funnel = pd.DataFrame(funnel_data)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_funnel['step'][::-1], df_funnel['support'][::-1],
               color=[WEB_COLORS.get(s, ZINC) for s in df_funnel['step'][::-1]],
               alpha=0.85, edgecolor='none', height=0.6)

for bar, row in zip(bars, df_funnel.iloc[::-1].itertuples()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{row.count} sesiones ({row.support:.0%})',
            va='center', fontsize=9, color='#e4e4e7')

ax.set_xlabel('Fracción de sesiones totales')
ax.set_title('Embudo de Conversión Web\n(% de sesiones que alcanzan cada etapa)')
ax.set_xlim(0, 1.3)
ax.grid(axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# ── Rutas al checkout ─────────────────────────────────────────────
checkout_paths = [
    (seq, ' → '.join(seq))
    for seq in web_db
    if 'checkout' in seq
]

# Agrupar rutas únicas
from collections import Counter
path_counter = Counter(path_str for _, path_str in checkout_paths)

print(f'🛒 RUTAS AL CHECKOUT ({len(checkout_paths)} sesiones con checkout)')
print('═' * 65)
for path, count in path_counter.most_common(10):
    pct = count / len(web_db)
    bar = '█' * int(pct * 40)
    print(f'  {count:3d} ({pct:.0%}) {bar}')
    print(f'         {path}')
    print()

---
## 10. Análisis de Negocios — Dataset B: Cross-sell Secuencial

En compras, los patrones secuenciales permiten:
- Diseñar **secuencias de email** post-compra
- Saber cuándo recomendar el producto B después de que el cliente compró A
- Medir la efectividad de **bundles dinámicos** a lo largo del tiempo

In [ ]:
# ── Recomendaciones de siguiente compra ──────────────────────────
def next_purchase_recommendations(db, seed_product, top_n=5):
    """Dado que el cliente acaba de comprar 'seed_product',
       ¿qué debería recomendarle a continuación?"""
    relevant = [seq for seq in db if seed_product in seq]
    next_counts = defaultdict(int)
    for seq in relevant:
        idx = seq.index(seed_product)
        if idx < len(seq) - 1:
            next_item = seq[idx + 1]
            if next_item != seed_product:
                next_counts[next_item] += 1
    total = len(relevant)
    recs = sorted(next_counts.items(), key=lambda x: -x[1])
    print(f'Seed: [{seed_product}]  ({len(relevant)} clientes que lo compraron)')
    print(f'Recomendaciones (siguiente compra inmediata):')
    for item, cnt in recs[:top_n]:
        conf = cnt / total
        print(f'  → {item:20s}  confianza inmediata = {conf:.0%}  ({cnt} clientes)')
    print()

for seed in ['laptop', 'phone', 'tablet']:
    next_purchase_recommendations(purchase_db, seed)
    print()

In [ ]:
# ── Confianza secuencial completa (no solo siguiente inmediato) ───
print('🎯 CONFIANZA SECUENCIAL (gap-free no requerido)')
print('   conf([A] ⟹ [B]) = % de clientes que compraron A y luego B (en cualquier momento)')
print('═' * 70)

seeds    = ['laptop', 'phone', 'tablet', 'charger']
targets  = ['mouse', 'keyboard', 'charger', 'case', 'earbuds', 'headphones']

conf_matrix = pd.DataFrame(index=seeds, columns=targets, dtype=float)
for s in seeds:
    for t in targets:
        conf_matrix.loc[s, t] = seq_confidence([s], t, purchase_db)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(conf_matrix.astype(float), ax=ax, cmap='YlOrRd', vmin=0, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 10},
            linewidths=0.3, linecolor='#18181b')
ax.set_title('Confianza Secuencial: [producto semilla] ⟹ [siguiente compra eventual]')
ax.set_xlabel('Siguiente compra (eventual)')
ax.set_ylabel('Compra semilla')
plt.tight_layout()
plt.show()

print()
print('Lectura: si el cliente compró "laptop" hay', end=' ')
v = conf_matrix.loc['laptop', 'mouse']
print(f'{v:.0%} de probabilidad de que compre "mouse" en alguna compra futura.')

In [ ]:
# ── Resumen ejecutivo ─────────────────────────────────────────────
print('📋 RESUMEN EJECUTIVO — RECOMENDACIONES DE NEGOCIO')
print('═' * 65)
print()
print('=== DATASET A: Navegación Web ===')
print()

# Drop-off más grande
worst_drop = max(range(1, len(funnel_data)), key=lambda i:
    1 - (funnel_data[i]['count'] / funnel_data[i-1]['count'] if funnel_data[i-1]['count'] > 0 else 0))
prev = funnel_data[worst_drop-1]
curr = funnel_data[worst_drop]
drop = 1 - curr['count'] / prev['count']
print(f'  ⚠️  Mayor caída en el embudo: {prev["step"]} → {curr["step"]} ({drop:.0%} drop-off)')
print(f'     Recomendación: revisar la UX de la página "{curr["step"]}"')
print()

top_path = path_counter.most_common(1)[0]
print(f'  ✅ Ruta de conversión más común:')
print(f'     {top_path[0]} ({top_path[1]} sesiones)')
print()

print('=== DATASET B: Historial de Compras ===')
print()

# Top secuencia de 2 por confianza en compras
best_cross_sell = []
for _, row in df_purch[df_purch['length'] == 2].iterrows():
    a, b = row['pattern']
    conf = seq_confidence([a], b, purchase_db)
    best_cross_sell.append({'pattern': row['pattern_str'], 'conf': conf, 'sup': row['support']})

best_cross_sell = sorted(best_cross_sell, key=lambda x: -x['conf'])[:5]
print('  🛒 Top 5 oportunidades de cross-sell secuencial:')
for i, item in enumerate(best_cross_sell, 1):
    print(f'     {i}. {item["pattern"]:35s} conf={item["conf"]:.0%}  sup={item["sup"]:.2f}')

---
## 11. Ejercicios

### 🔀 Ejercicio 1 — El orden importa
Compara el soporte de `[home → products → cart]` con `[cart → products → home]`. ¿Son iguales? ¿Por qué en AR darían el mismo soporte pero en SM no?

### 📉 Ejercicio 2 — Umbral de soporte
Para el Dataset A, baja `MIN_SUPPORT_WEB` de 0.30 a 0.15. ¿Cuántos patrones nuevos aparecen? ¿Encuentras patrones de 4+ eventos que sean accionables desde negocio?

### 🌐 Ejercicio 3 — Análisis de embudo
¿Qué páginas tienen el mayor drop-off en el Dataset A? Si fueras analista de UX, ¿qué experimento A/B diseñarías para reducir esa caída?

### 🔗 Ejercicio 4 — Cross-sell por timing
Usando el Dataset B, calcula la diferencia entre **confianza inmediata** (siguiente compra directa) y **confianza secuencial** (cualquier compra futura) para `laptop → keyboard`. ¿Cuál es más útil para una campaña de email en D+7 tras la compra?

### 🎯 Ejercicio 5 — Tu propio dataset
Modifica `NAVIGATION_PATTERNS` para simular un **flujo de una app móvil de fitness** (onboarding → workout → stats → share → premium) y analiza los caminos hacia la conversión a premium.